<a href="https://colab.research.google.com/github/alimor37/WAAM_Weld_ML/blob/main/Phase4_HyperparameterTuning_RF%26GBR_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hyperparameter Tuning of randomforest and gradient boosting regressor for voing regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import LeaveOneOut, GridSearchCV, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings('ignore')

# =====================================================================
# 1. LOAD DATA & PREPROCESSING
# =====================================================================
github_url = "https://raw.githubusercontent.com/alimor37/WAAM_Weld_ML/main/Cleaned_data.xlsx"
df = pd.read_excel(github_url)
target_rp = 'Rp0.2_Mpa'
target_rm = 'Rm_Mpa'

df[target_rp] = pd.to_numeric(df[target_rp], errors='coerce')
df[target_rm] = pd.to_numeric(df[target_rm], errors='coerce')

if df['t85_s'].dtype == object:
    df['t85_s'] = df['t85_s'].astype(str).apply(
        lambda x: sum([float(i) for i in x.split('-')])/2 if '-' in x else float(x) if x.replace('.', '', 1).isdigit() else None
    )

trace_elements = ['Nb', 'B', 'O', 'Ti', 'Al', 'S', 'P', 'Mn', 'C', 'Si', 'Cr', 'Ni', 'Mo', 'V', 'Cu', 'N']
for col in trace_elements:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace('<', '').str.strip(), errors='coerce')

bead_col = 'No_of_Beads' if 'No_of_Beads' in df.columns else 'No_of_Bead'
if 'SinglePass' not in df.columns:
    df['SinglePass'] = ((df[bead_col] == 1) | (df['Weld_Method'].astype(str).str.strip().isin(['Rapid Arc', 'Laser-Hybrid']) if 'Weld_Method' in df.columns else False)).astype(int)
df['Base1100'] = df['Base_Material'].astype(str).apply(lambda x: 1 if '1100' in str(x) else 0)

# =====================================================================
# 2. HYBRID FEATURES
# =====================================================================
df['Mn_temp'] = df['Mn'].fillna(df['Mn'].mean())
df['S_temp'] = df['S'].fillna(df['S'].mean())
df['Ti_temp'] = df['Ti'].fillna(df['Ti'].mean())
df['B_temp'] = df['B'].fillna(df['B'].mean())

df['Pcm_t85_Synergy'] = df['Pcm'] * df['t85_s']
df['Mn_S_Ratio'] = df['Mn_temp'] / (df['S_temp'] + 1e-5)
df['Ti_B_Synergy'] = df['Ti_temp'] * df['B_temp']

# استفاده از ویژگی‌های طلایی استخراج شده توسط RFECV
rp_golden_features = ['t85_s', bead_col, 'SinglePass', 'Base1100', 'C', 'Si', 'Mn', 'P', 'Cr', 'Ni', 'Mo', 'V', 'Nb', 'Cu', 'Ti', 'B', 'N']
rp_features = [col for col in rp_golden_features if col in df.columns]

rm_golden_features = ['Pcm', 't85_s', bead_col, 'Dilution_Ni_pct', 'SinglePass', 'Base1100', 'C', 'Si', 'Mn', 'P', 'S', 'Cr', 'Ni', 'Mo', 'V', 'Nb', 'Cu', 'Al', 'Ti', 'B', 'O', 'N', 'Pcm_t85_Synergy', 'Mn_S_Ratio', 'Ti_B_Synergy']
rm_features = [col for col in rm_golden_features if col in df.columns]

loo = LeaveOneOut()

# =====================================================================
# 3. TUNING ENGINE
# =====================================================================
def tune_tree_models(target_col, model_name, model, param_grid, features):
    print(f"{'='*65}")
    print(f"🚀 TUNING {model_name} FOR: {target_col}")
    print(f"{'='*65}")

    valid_df = df.dropna(subset=['Pcm', 't85_s', target_col]).copy()
    X = valid_df[features].values
    y = valid_df[target_col].values

    # درخت‌ها نیازی به اسکیل ندارند، اما برای یکدستی ساختار، از همان پایپ‌لاین استفاده می‌کنیم
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', MinMaxScaler()),
        ('regressor', model)
    ])

    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=loo,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)
    best_pipe = grid_search.best_estimator_

    print(f"\n✅ BEST PARAMETERS FOUND:")
    for param, value in grid_search.best_params_.items():
        print(f"   - {param.replace('regressor__', '')}: {value}")

    y_pred = cross_val_predict(best_pipe, X, y, cv=loo)
    best_r2 = r2_score(y, y_pred)
    best_rmse = np.sqrt(mean_squared_error(y, y_pred))

    print(f"\n🏆 OPTIMIZED RESULTS:")
    print(f"   - LOOCV R²:   {best_r2:.3f}")
    print(f"   - LOOCV RMSE: {best_rmse:.1f} MPa")
    print("="*65 + "\n")

# =====================================================================
# 4. EXPANDED PARAMETER GRIDS
# =====================================================================

rf_grid = {
    'regressor__n_estimators': [130, 140, 150, 160, 170],
    'regressor__max_depth': [None, 5, 7, 9, 12],
    'regressor__min_samples_split': [2, 3],
    'regressor__min_samples_leaf': [1, 2],
    'regressor__max_features': ['sqrt', 3, 4, 5, 6]
}

gbr_grid = {
    'regressor__n_estimators': [50, 100, 150, 200, 250, 300],
    'regressor__learning_rate': [0.01, 0.05, 0.1, 0.5, 1],
    'regressor__max_depth': [1, 2, 3, 4], # عمق کم برای جلوگیری از Overfitting
    'regressor__min_samples_leaf': [1, 2, 3, 4],
    'regressor__subsample': [0.2, 0.4, 0.6, 0.8, 1.0],
    'regressor__max_features': ['sqrt', 'log2', None]
}

# =====================================================================
# 5. EXECUTION
# =====================================================================
# تیونینگ RF برای استحکام تسلیم (Rp0.2)
tune_tree_models(target_rp, 'Random Forest', RandomForestRegressor(random_state=42), rf_grid, rp_features)

# تیونینگ GBR برای استحکام کششی (Rm)
# tune_tree_models(target_rm, 'Gradient Boosting', GradientBoostingRegressor(random_state=42), gbr_grid, rm_features)

🚀 TUNING Random Forest FOR: Rp0.2_Mpa
Fitting 28 folds for each of 500 candidates, totalling 14000 fits

✅ BEST PARAMETERS FOUND:
   - max_depth: 9
   - max_features: sqrt
   - min_samples_leaf: 1
   - min_samples_split: 2
   - n_estimators: 150

🏆 OPTIMIZED RESULTS:
   - LOOCV R²:   0.664
   - LOOCV RMSE: 62.3 MPa

